In [ ]:
%pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

In [ ]:
%pip install langchainhub

previously, it was *from langchain import hub* but now *hub* was removed from core *langchain* package. so, now it became. *import lanchainhub as hub*

In [ ]:
%pip install langchain_text_splitters

Previously, it was *from langchain.text_splitter import RecursiveCharacterTextSplitter* . Now, *text_splitter* is removed from core *langchain* package and to its own package *langchain_text_splitters*

In [11]:
import os
os.environ['LANGCHAIN_TRACKING_V2'] ='true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'

In [ ]:
%pip install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")

In [ ]:

# Force reload
load_dotenv(override=True)


# Check where Python is looking for .env
print("\n=== FILE LOCATION ===")
print("Current working directory:", os.getcwd())

# Check if .env file actually exists here
env_path = os.path.join(os.getcwd(), ".env")
print(".env exists at this path:", os.path.exists(env_path))

In [ ]:
%pip install --upgrade langchain langchainhub

In [ ]:
%pip install langsmith

In [27]:
from langsmith import Client

In [28]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

WebBaseLoader → bs4 cleans HTML → RecursiveCharacterTextSplitter chunks it
→ OpenAIEmbeddings converts chunks → Chroma stores them
→ User asks question → Chroma retrieves relevant chunks
→ RunnablePassthrough passes question through → hub.pull provides the prompt
→ ChatOpenAI generates answer → StrOutputParser returns clean text.



When WebBaseLoader makes HTTP requests to websites, it sends a User-Agent header identifying who is making the request. Right now it's sending a generic/blank identifier, which some websites block or flag as suspicious bot traffic.
It won't break anything if you ignore it, but setting it is good practice — especially if you're scraping sites that enforce bot protection.

In [18]:
import os
os. environ['USER_AGENT'] ='MyRAGapp/1.0'

In [30]:
#### INDEXING ####

# Load Documents
loader = WebBaseLoader(
    web_paths= ("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header")
        )
    ),
)
docs=loader.load()

#split
text_splitter= RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits=text_splitter.split_documents(docs)

#Embed
vector_store=Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever=vector_store.as_retriever()

#### RETRIEVAL AND GENERATION ####

# prompt
client = Client()
prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

# LLM
llm=ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# Post-processing
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)


# Chain
rag_chain = (
    {"context":retriever | format_docs,"question":RunnablePassthrough()}
    | prompt
    | llm
    |StrOutputParser()
)

# Question
rag_chain.invoke('what is Task Decomposition')

'Task decomposition is a technique used to break down complex tasks into smaller and simpler steps, making them more manageable. This approach involves transforming big tasks into multiple smaller tasks to enhance model performance and interpretation of the thinking process. Different methods like Chain of Thought and Tree of Thoughts are used to decompose tasks into manageable steps.'

***temperature=0*** means deterministic — no randomness, same question always gives same answer (good for factual Q&A)